---
title: 'ATP Tennis Data: Import and Basic Cleaning'
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.17.3
  kernelspec:
    display_name: Python 3
    language: python
    name: python3
---


## 1. Imports and Settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)

START_YEAR = 2010
END_YEAR = 2025

DATA_URL = "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{year}.csv"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Download and Load the Data

This downloads the yearly ATP files. If the file already exists locally, the notebook uses the local copy.

In [ ]:
all_years = []

for year in range(START_YEAR, END_YEAR + 1):
    local_file = RAW_DIR / f"atp_matches_{year}.csv"
    url = DATA_URL.format(year=year)

    if local_file.exists():
        year_data = pd.read_csv(local_file)
    else:
        year_data = pd.read_csv(url)
        year_data.to_csv(local_file, index=False)

    year_data["source_year"] = year
    all_years.append(year_data)

matches = pd.concat(all_years, ignore_index=True)

print(matches.shape)
matches.head()

## 3. Keep the Columns We Need

We are keeping basic match information and pre-match information like ranking, ranking points, and age. We are not using same-match serving stats yet because those would not be known before predicting a match.

In [ ]:
columns_to_keep = [
    "tourney_id",
    "tourney_name",
    "surface",
    "tourney_date",
    "match_num",
    "best_of",
    "round",
    "winner_id",
    "winner_name",
    "winner_age",
    "loser_id",
    "loser_name",
    "loser_age",
    "winner_rank",
    "winner_rank_points",
    "loser_rank",
    "loser_rank_points",
    "source_year",
]

matches = matches[columns_to_keep].copy()

matches.head()

## 4. Basic Cleaning

In [ ]:
matches["match_date"] = pd.to_datetime(
    matches["tourney_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

number_columns = [
    "winner_age",
    "loser_age",
    "winner_rank",
    "loser_rank",
    "winner_rank_points",
    "loser_rank_points",
]

for column in number_columns:
    matches[column] = pd.to_numeric(matches[column], errors="coerce")

cleaned = matches.dropna(
    subset=["match_date", "winner_id", "loser_id", "winner_rank", "loser_rank"]
).copy()

print("Original rows:", len(matches))
print("Rows after basic cleaning:", len(cleaned))
cleaned.head()

## 5. Starter Feature Engineering

These features are simple and easy to explain. Since the data is still in winner/loser format, these are mostly useful for early exploration and for building a ranking baseline later.

In [ ]:
cleaned["ranking_diff"] = cleaned["winner_rank"] - cleaned["loser_rank"]
cleaned["rank_points_diff"] = cleaned["winner_rank_points"] - cleaned["loser_rank_points"]
cleaned["age_diff"] = cleaned["winner_age"] - cleaned["loser_age"]
cleaned["higher_ranked_won"] = (cleaned["winner_rank"] < cleaned["loser_rank"]).astype(int)
cleaned["match_year"] = cleaned["match_date"].dt.year

cleaned[[
    "match_date",
    "surface",
    "winner_name",
    "loser_name",
    "winner_rank",
    "loser_rank",
    "ranking_diff",
    "higher_ranked_won",
]].head()

## 6. Quick Checks

In [ ]:
cleaned.isna().mean().sort_values(ascending=False).head(10)

In [ ]:
cleaned.groupby("match_year").size().tail()

In [ ]:
cleaned["surface"].value_counts(dropna=False)

In [ ]:
higher_ranked_win_rate = cleaned["higher_ranked_won"].mean()
print(f"Higher-ranked player won {higher_ranked_win_rate:.1%} of matches.")

## 7. Simple Visualization

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(cleaned["ranking_diff"], bins=50)
plt.title("Ranking Difference: Winner Rank - Loser Rank")
plt.xlabel("Ranking Difference")
plt.ylabel("Number of Matches")
plt.show()

## 8. Save Processed Files

In [ ]:
combined_path = PROCESSED_DIR / "atp_matches_combined.csv"
cleaned_path = PROCESSED_DIR / "atp_matches_cleaned_starter.csv"

matches.to_csv(combined_path, index=False)
cleaned.to_csv(cleaned_path, index=False)

print("Saved:", combined_path)
print("Saved:", cleaned_path)